# Data Preparation

The notebook will show all the options available for preparing the data before sending it to the model.

In [2]:
import numpy as np

from sckitflow.data import DataManager
from sckitflow.data.sim import get_dummy_adata
from sckitflow.dataset.toy_data import get_toy_dataset
from sklearn.decomposition import PCA

## Preliminaries

### Dummy Datasets

`sckitflow` allows you to create different types of dummy datasets using `sklearn.datasets` and wraps it in `Anndata` format using the function `get_toy_dataset(name, **kwargs)` where `name` refers to the type of dataset to be created and `**kwargs` are the additional parameters specific to dataset type. 

The following code block shows all the possible dataset types for creating dummy data

In [5]:
blobs_data = get_toy_dataset("blobs").adata
checkerboard_data = get_toy_dataset("checkerboard").adata
circles_data = get_toy_dataset("circles").adata
moons_data = get_toy_dataset("moons").adata
s_curve_data = get_toy_dataset("s_curve").adata
swiss_roll_data = get_toy_dataset("swiss_roll").adata

Investigating the data, we can see that the anndata structure varies in terms of fields with each data type. However, all of them have `adata.uns['dataset_info]` which contains all the parameters used in dataset creation. The default parameters remain the same from `sklearn.datasets` default values.

In [6]:
print(blobs_data)
print(checkerboard_data)

AnnData object with n_obs × n_vars = 1000 × 2
    uns: 'dataset_info'
    obsm: 'Y'
AnnData object with n_obs × n_vars = 1000 × 2
    uns: 'dataset_info'
    obsm: 'row_cluster'
    varm: 'col_cluster'


In [7]:
blobs_data.uns["dataset_info"]

{'name': 'blobs',
 'random_state': 42,
 'No of centers': 3,
 'cluster_std': 1.0,
 'center_box': (-10.0, 10.0),
 'shuffle': True,
 'centers': {'0': array([-2.50919762,  9.01428613]),
  '1': array([4.63987884, 1.97316968]),
  '2': array([-6.87962719, -6.88010959])}}

### PCA Representation

Preprocessing such as PCA is **not** performed by `sckitflow` — the caller computes the representation with their tool of choice and stores it in `.obsm`. Here we use scikit-learn's `PCA` and later point `DataManager` at it via `sample_rep`. Fitted attributes such as `pca.components_`, `pca.explained_variance_`, `pca.mean_`, and `pca.n_components_` live on the estimator.

In [8]:
blobs_data = get_toy_dataset("blobs", n_features=200, centers=5).adata

pca = PCA(n_components=50)
blobs_data.obsm["X_pca"] = pca.fit_transform(blobs_data.X)

print(f"Original Shape: {blobs_data.X.shape}")
print(f"New Shape: {blobs_data.obsm['X_pca'].shape}")

Original Shape: (1000, 200)
New Shape: (1000, 50)


## DataManager

### Basic Initialization

Creating Dummy data with 5000 samples and 200 features

In [ ]:
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
pca = PCA(n_components=50)
dummy_data.obsm["X_pca"] = pca.fit_transform(dummy_data.X)
print(dummy_data)
print(dummy_data.obsm["X_pca"].shape)

AnnData object with n_obs × n_vars = 5000 × 200
    uns: 'dataset_info'
    obsm: 'Y', 'X_pca'
(5000, 50)


Initializing `DataManager` and reading the data through it. `get_distribution_data` applies the schema to an `AnnData` and returns the containers the model is sized from.

In [ ]:
dm = DataManager()
distr = dm.get_distribution_data(dummy_data)
print(distr)

Initializing DataManager with the PCA representation and compiling the data. `DataManager` checks for the key in `.obsm`. It defaults to `.X` if key not found.

In [ ]:
dm1 = DataManager(sample_rep="X_pca")
distr_pca = dm1.get_distribution_data(dummy_data)
print(distr_pca)

### Advanced Initialization

The initialization of `Datamanager` can be classified into 5 groups -

* Group Data
* State Data
* Coupling Data
* Condition Data
* Response Data

#### Group Data

This group of parameters control how the oberservation metadata is turned into numerical encoding. The parameters are -

* `groups`: Collection of `.obs` identifiers used to define grouping.


* `groups_reps`: Dictionary mapping `.obs` identifiers to pre defined numerical encoding which are stored in `.uns`


* `groups_encoding`: This parameter is used if the encoding have not been created and stored in `.uns` and we want to create the encoding now. The input is a dictionary mapping the `groups` identifiers to an encoder. Each group column takes either `groups_reps` or `groups_encoding` — never both.


* __NOTE__ The strings `"label"` and `"one-hot"` are shorthand for the two parameter-free encoders. For anything parameterized, pass an encoder instance from `sckitflow.data.group_encoders` (`from sckitflow.data import group_encoders`) — `Label`, `OneHot`, `Identity`, `Log1p`, `Affine`. These are frozen dataclasses of plain scalars rather than callables, which is what lets a whole `DataManager` be serialized. Use them to pin a vocabulary, as in `OneHot(categories=("control", "drugA"))`, so a round-tripped config reproduces identical columns; unknown categories then raise instead of being silently dropped.

In [22]:
# Creating Dummy data
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata

# Creating dummy .obs columns
drug_labels = ["drugA", "control"]
ko_labels = ["koA", "control"]
dummy_data.obs["drug"] = np.random.choice(drug_labels, size=dummy_data.n_obs)
dummy_data.obs["ko"] = np.random.choice(ko_labels, size=dummy_data.n_obs)

In [ ]:
dm = DataManager(
    groups=["drug", "ko"],
    groups_encoding={"drug": "one-hot", "ko": "label"},
)
distr = dm.get_distribution_data(dummy_data)
print(distr)

The schema does not split the data itself -- it *describes* it. The grouping becomes visible when the
data is **streamed**: `get_eval_loader` walks each unique `groups` combination once and yields the batch
together with its `leaf`, the tuple of group values it came from. Here that is the 4 combinations
`(drugA, koA)`, `(control, koA)`, `(drugA, control)`, `(control, control)`.

In [ ]:
eval_loader = dm.get_eval_loader(dummy_data)

print(f"{len(eval_loader)} groups, ordered as {eval_loader.group_cols}")
for _, leaf in eval_loader:
    print(leaf)

In [ ]:
# One group's batch: `leaf` says which group it is, `target_state` holds that group's cells.
step_data, leaf = next(iter(dm.get_eval_loader(dummy_data)))
print(leaf, step_data["target_state"].shape)

We can use `DataManager.groups_data_schema` to look at our variables

In [26]:
print(dm.groups_data_schema.groups)
print(dm.groups_data_schema.groups_encoders)
print(dm.groups_data_schema.groups_reps)

['drug', 'ko']
{'drug': OneHot(categories=None), 'ko': Label(classes=None)}
{}


#### State Data

This is just the basic initialization. The only parameter here is `sample_rep` by which you can choose which representation you want to use as input. The str value will be searched in `.obs` and if not found, it reverts to `.X`

In [ ]:
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
pca = PCA(n_components=50)
dummy_data.obsm["X_pca"] = pca.fit_transform(dummy_data.X)

dm = DataManager(sample_rep="X_pca")
distr = dm.get_distribution_data(dummy_data)
print(distr)

In [19]:
dm.state_data_schema.sample_rep

'X_pca'

#### Coupling Data

This group of parameters control how the source and the target data distribution is coupled. The parameters are-

1. `sample_rep`: To choose the sourse representation

2. `target_rep`: To choose the target representation

3. `n_shared_dims`: If for some reason, the source and target embedding is not in the same latent space, then we have the option to choose the first n dimesions from both embeddings to sort of create a latent space.

In [3]:
n_obs_pert = 5000
n_obs_ctrl = 200
adata = get_dummy_adata(n_obs_pert=n_obs_pert, n_obs_ctrl=n_obs_ctrl)

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3312.0_x64__qbz5n2kfra8p0\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3312.0_x64__qbz5n2kfra8p0\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [ ]:
dm = DataManager(
    sample_rep="X_tgt",
    source_rep="X_src",
    n_shared_dims=10,
)
distr = dm.get_distribution_data(adata)
distr

In [7]:
print(dm.coupling_data_schema.source_rep)
print(dm.coupling_data_schema.target_rep)
print(dm.coupling_data_schema.n_shared_dims)
print(dm.coupling_data_schema.has_incomparable_spaces)

X_src
X_tgt
10
True


#### Condition Data

In [27]:
# regenerate data with a categorical condition column for this section
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
dummy_data.obs["drug"] = np.random.choice(["drugA", "control"], size=dummy_data.n_obs)
dummy_data

AnnData object with n_obs × n_vars = 5000 × 200
    obs: 'drug', 'ko'
    uns: 'dataset_info'
    obsm: 'Y'

In [ ]:
dm = DataManager(target_categorical_covs_dict={"drug": "one-hot"})
distr = dm.get_distribution_data(dummy_data)

In [ ]:
dm1 = DataManager()
distr_plain = dm1.get_distribution_data(dummy_data)

In [ ]:
distr

In [ ]:
distr_plain

#### Response Data

## Streaming to the model

Configuring the schema is only half of it -- the model never sees an `AnnData`, it sees batches. A **splitter** writes a `split` column into `.obs`, and `DataManager.get_dataloaders` turns that into one streaming loader per split. Splitting policy and data configuration stay separate: the splitter only labels observations, and the data manager only reads them.

In [ ]:
from sckitflow.data import CombinationSplitter

dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
dummy_data.obs["drug"] = np.random.choice(["drugA", "drugB", "control"], size=dummy_data.n_obs)
dummy_data.obs["ko"] = np.random.choice(["koA", "koB"], size=dummy_data.n_obs)

# Categorical conditions are encoded through `.uns`, keyed by the condition value.
dummy_data.uns["drug"] = {v: np.random.randn(1, 8) for v in dummy_data.obs["drug"].unique()}

# Whole (drug, ko) combinations are held out, and every `ko` keeps at least one combination
# in train. Control rows are labelled separately -- they are the shared source, never a split.
splitter = CombinationSplitter(
    group_keys=["drug", "ko"],
    always_train_keys=["ko"],
    control_key="drug",
    control_value="control",
    test_fraction=0.25,
)
dummy_data = splitter(dummy_data)
dummy_data.obs["split"].value_counts()

`get_dataloaders` returns `{split: loader}`. Controls are shared across splits rather than being a split of their own, so a control-only label produces no loader of its own.

In [ ]:
dm = DataManager(
    groups=["ko"],
    conditions={"drug": ["drug"]},
    conditions_reps={"drug": "drug"},
    groups_encoding={"ko": "label"},
    control_values_dict={"drug": "control"},
)

loaders = dm.get_dataloaders(dummy_data, split_by="split", batch_size=128)
print({split: f"{len(loader)} batches" for split, loader in loaders.items()})

Each loader yields a ready `StepData`: the perturbed cells as `target_state`, their matched controls as `source_state`, and the group/condition encodings tiled to the batch.

In [ ]:
step_data = next(iter(loaders["train"]))

print("target:", step_data["target_state"].shape)
print("source:", step_data["source_state"].shape)
print("conditions:", {k: tuple(v.shape) for k, v in (step_data["target_condition_data"] or {}).items()})
print("groups:", {k: tuple(v.shape) for k, v in (step_data["target_group_data"] or {}).items()})